# Checking a Hand-Built Autograd Against PyTorch

**Session 19–20 · companion for HW 3**

HW 3 Part A asks you to write a scalar autograd engine: `__add__`, `__mul__`,
`__pow__`, `relu`, and a `backward()` that topologically sorts the graph and
accumulates gradients in reverse. This notebook implements none of that — it
gives you the **oracle** you check it against.

PyTorch already computes these gradients correctly. If your `Value` and
`torch.tensor` disagree on the same expression, your engine is wrong, and the
expression that disagrees tells you which rule to look at.

Run every cell; the outputs are real runs.

In [1]:
import torch

def torch_grads(fn, **inputs):
    """Evaluate fn with torch tensors and return (value, {name: gradient})."""
    tensors = {k: torch.tensor(float(v), requires_grad=True) for k, v in inputs.items()}
    out = fn(**tensors)
    out.backward()
    return out.item(), {k: t.grad.item() for k, t in tensors.items()}


value, grads = torch_grads(lambda a, b: a * b + a, a=3.0, b=-2.0)
print("f(a,b) = a*b + a   at a=3, b=-2")
print("  value", value)
print("  d/da ", grads["a"], "  (should be b + 1 = -1)")
print("  d/db ", grads["b"], "  (should be a = 3)")

f(a,b) = a*b + a   at a=3, b=-2
  value -3.0
  d/da  -1.0   (should be b + 1 = -1)
  d/db  3.0   (should be a = 3)


## The expressions worth testing

Each of these isolates one rule. Test them in order: a failure in the first
makes every later failure meaningless.

In [2]:
cases = {
    "add          a + b":            (lambda a, b: a + b,              dict(a=3.0, b=-2.0)),
    "mul          a * b":            (lambda a, b: a * b,              dict(a=3.0, b=-2.0)),
    "reuse        a * a":            (lambda a: a * a,                 dict(a=3.0)),
    "chain        (a + b) * a":      (lambda a, b: (a + b) * a,        dict(a=3.0, b=-2.0)),
    "pow          a ** 3":           (lambda a: a ** 3,                dict(a=2.0)),
    "div          a / b":            (lambda a, b: a / b,              dict(a=3.0, b=-2.0)),
    "neg/sub      a - b":            (lambda a, b: a - b,              dict(a=3.0, b=-2.0)),
    "relu on +    relu(a)":          (lambda a: torch.relu(a),         dict(a=2.0)),
    "relu on -    relu(a)":          (lambda a: torch.relu(a),         dict(a=-2.0)),
    "relu at 0    relu(a)":          (lambda a: torch.relu(a),         dict(a=0.0)),
    "deep         relu(a*b + a)*b":  (lambda a, b: torch.relu(a * b + a) * b, dict(a=3.0, b=2.0)),
}

for name, (fn, inputs) in cases.items():
    val, g = torch_grads(fn, **inputs)
    shown = "  ".join(f"d/d{k}={v:+.4f}" for k, v in g.items())
    print(f"{name:<26} value={val:+8.4f}   {shown}")

add          a + b         value= +1.0000   d/da=+1.0000  d/db=+1.0000
mul          a * b         value= -6.0000   d/da=-2.0000  d/db=+3.0000
reuse        a * a         value= +9.0000   d/da=+6.0000
chain        (a + b) * a   value= +3.0000   d/da=+4.0000  d/db=+3.0000
pow          a ** 3        value= +8.0000   d/da=+12.0000
div          a / b         value= -1.5000   d/da=-0.5000  d/db=-0.7500
neg/sub      a - b         value= +5.0000   d/da=+1.0000  d/db=-1.0000
relu on +    relu(a)       value= +2.0000   d/da=+1.0000
relu on -    relu(a)       value= +0.0000   d/da=+0.0000
relu at 0    relu(a)       value= +0.0000   d/da=+0.0000
deep         relu(a*b + a)*b value=+18.0000   d/da=+6.0000  d/db=+15.0000


### The three that catch most bugs

**`a * a`** — the same node used twice. A `backward()` that *assigns* instead of
*accumulating* gets `d/da = 3` here instead of `6`. This is the single most
common error in a first autograd, and no other expression reveals it.

**`relu` at exactly 0** — PyTorch reports `0.0`. There is no true derivative at
the kink, so this is a convention, and the tests expect you to match it.

**`relu(a*b + a) * b`** — a graph deep enough that a topological-sort bug shows
up, and note the inputs are `a=3, b=2`, not the `b=-2` used above. With `b=-2`
the relu input is negative, the whole expression is zero, and *every* gradient
is zero — a test that passes for a broken engine as readily as a correct one.
Pick inputs that keep the relu open, or you are testing nothing.

## How to use this against your own engine

Once your `Value` class exists, the comparison is four lines. Keep it in a
scratch file while you work:

```python
from autograd import Value

a, b = Value(3.0), Value(-2.0)
out = (a * b + a)
out.backward()
print(a.grad, b.grad)      # compare against the table above
```

A gradient that is right in magnitude and wrong in sign points at `__sub__` or
`__neg__`. A gradient that is exactly half or exactly double points at
accumulation — you are overwriting or double-counting a path. A gradient of
`0.0` where the table says otherwise usually means the node never got visited.

## Part B: what `nn.Module` gives you back

Part B rebuilds the same idea with PyTorch doing the calculus. The shape of the
training loop is the part to get right, and these are the pieces the tests check.

In [3]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(4, 16), nn.ReLU(),
    nn.Linear(16, 3),
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable parameters:", n_params)
print("  layer 1:", 4 * 16 + 16, " layer 2:", 16 * 3 + 3)

trainable parameters: 131
  layer 1: 80  layer 2: 51


`count_parameters` is that one-liner. Note `requires_grad` in the filter: a
frozen backbone (Session 23) has parameters that are *not* trainable, and the
same function has to keep telling the truth then.

In [4]:
x = torch.randn(8, 4)
y = torch.randint(0, 3, (8,))

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

before = loss_fn(model(x), y).item()

optimizer.zero_grad()                 # 1. clear last step's gradients
loss = loss_fn(model(x), y)           # 2. forward
loss.backward()                       # 3. backward
optimizer.step()                      # 4. update

after = loss_fn(model(x), y).item()
print(f"loss {before:.4f} -> {after:.4f}")

loss 1.1106 -> 1.0630


Those four lines in that order are the whole training step. The one that gets
forgotten is `zero_grad()`: PyTorch **accumulates** into `.grad` — the same
design decision your `Value.backward()` has to make — so skipping it silently
sums this batch's gradient onto the last one's.

`CrossEntropyLoss` takes **logits**, not probabilities. Adding a `softmax` before
it applies the operation twice, and the model trains slowly instead of failing
loudly.

In [5]:
# HW 3 asks you to write `accuracy(model, loader)`, so this is not that
# function -- it is the three habits the tests check for, run inline.

model.eval()                              # 1. eval mode
with torch.no_grad():                     # 2. no graph for work you discard
    correct = (model(x).argmax(dim=1) == y).sum().item()
model.train()                             # 3. put it back

print("correct:", correct, "of", len(y))

correct: 3 of 8


Three habits there, and the tests check for all of them: `torch.no_grad()` so
evaluation does not build a graph it will never use, and `model.eval()` /
`model.train()` around it — which changes nothing for this model and everything
once dropout or batch norm appear in Session 21. Forgetting to switch back to
`train()` is the bug that makes the next epoch mysteriously stop learning.

---

## Where to go next

- **HW 3 Part A** — check every row of the table above against your `Value`.
- **HW 3 Part B** — the four-line step and the two habits are the shape the tests
  expect.
- **Reading, Session 19** — where these rules come from.